In [ ]:
import os
import json
import numpy as np
import torch
import cv2
from torch_geometric.data import DataLoader
from torch.utils.data import random_split
from deeplsd.models.deeplsd_inference import DeepLSD
import ground_truth.feature_extraction as ft
from ground_truth.visualization import plot_images, plot_coplanar_lines, plot_lines_bool
import matplotlib.pyplot as plt

import torch.nn as nn
import torch_geometric.nn as pyg_nn

# from models.dataset_inductive import GraphDatasetInductive
from models.model_utils import train_inductive, test_inductive, run_inference

In [ ]:
import torch 


from lightning_tools.lightning_gluestick_inspired import *
def run_inference_lightning(
    ckpt_path: str,
    data_loader: torch.utils.data.DataLoader,
    device: torch.device = torch.device('cpu'),
    threshold_structural: float = 0.5,
    threshold_coplanarity: float = 0.5,
):
    # 1) load LightningModule (this also restores hparams)
    model = AttentionBothCoplanar.load_from_checkpoint(ckpt_path)
    #print(torch.exp(-2*model.log_sigma_node),torch.exp(-2*model.log_sigma_edge))
    model = model.to(device).eval()
    print("→ trained roi_align_embedding_shape:", model.hparams.roi_align_embedding_shape)
    print("→ channels_conv_roi_embedding:", model.hparams.channels_conv_roi_embedding)
    print("→ in_channels_DeepLSD:",       model.hparams.in_channels_DeepLSD)
    print("→ expected fuse input dim:",   model.hparams.channels_conv_roi_embedding + model.hparams.in_channels_DeepLSD)
    all_node_preds = []
    all_edge_preds = []

    with torch.no_grad():
        for batch in data_loader:
            batch = batch.to(device)
            # 2) forward returns raw logits
            node_logits, edge_logits = model(batch)

            # 3) turn into probs
            node_probs = torch.sigmoid(node_logits)    # shape [B, N, 1] or [N,1]
            edge_probs = torch.sigmoid(edge_logits)    # shape [B, E, 1] or [E,1]

            # 4) threshold to get binary labels
            node_labels = (node_probs >= threshold_structural).float()
            edge_labels = (edge_probs >= threshold_coplanarity).float()

            all_node_preds.append(node_labels.cpu())
            all_edge_preds.append(edge_labels.cpu())

    return all_node_preds, all_edge_preds


In [ ]:
import os
import json
import orjson

import numpy as np
import torch
import cv2
from torch_geometric.data import Data, Dataset
# from notebooks.models.dataset_utils import extract_line_feature_ROIAlign,sample_lines_grid
from sklearn.neighbors import NearestNeighbors

from typing import Optional
import logging
    # -------------------------------------------
def line_geometry(line_pts: torch.Tensor):
    """
    line_pts : [N, 2, 2]  (x1,y1,x2,y2 per line)
    returns   :  ϕ_node   [N, 5]
    [mid_x, mid_y, dir_x, dir_y, length]
    """
    p1, p2   = line_pts[:, 0], line_pts[:, 1]           # [N,2]  [N,2]
    mid      = 0.5 * (p1 + p2)                          # [N,2]
    vec      = p2 - p1
    length   = vec.norm(dim=1, keepdim=True)            # [N,1]
    dir_norm = F.normalize(vec, dim=1)                  # [N,2]
    return torch.cat([mid, dir_norm, length], dim=1)    # [N,5]

def _load_image(filepath: str, color_conversion: Optional[int] = None) -> Optional[np.ndarray]:
    """Loads an image using OpenCV."""
    if not os.path.exists(filepath):
        logging.error(f"Image file not found: {filepath}")
        return None
    try:
        img = cv2.imread(filepath, cv2.IMREAD_UNCHANGED) # Load as is (handles color, grayscale, alpha)
        if img is None:
            logging.error(f"Failed to load image (cv2.imread returned None): {filepath}")
            return None
        if color_conversion is not None:
            img = cv2.cvtColor(img, color_conversion)
        return img
    except Exception as e:
        logging.error(f"Error loading image {filepath}: {e}")
        return None



import os, json, logging
import numpy as np
import cv2
import torch
from torch.utils.data import Dataset
from torch_geometric.data import Data
from lightning_tools.line_sampler import LineSampler  # <-- your LightningModule
from lightning_tools.line_sampler import extract_line_feature_ROIAlign

class GraphDatasetInference(Dataset):
    def __init__(self, embeddings, image_path, coords, roi_output_size=(64, 64), method="sample", device=None):
        super().__init__()
                
        self.embeddings = embeddings
        self.image_path = image_path
        self.coords = coords
        self.roi_output_size = roi_output_size
        self.method = method
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Only instantiate the sampler if we're going to use it
        if self.method == "sample":
            num_samples, width = self.roi_output_size
            self.sampler = LineSampler(
                num_samples=num_samples,
                width=width
            ).to(self.device)

    def __len__(self):
        return len(self.image_path)

    def __getitem__(self, idx):
        # load JSON
 
        file_path_img    = self.image_path[idx]

        # build node embeddings + labels
        feats, line_coords = self.embeddings[idx], self.coords[idx]
    

        x_emb = torch.tensor(np.vstack(np.array(feats)), dtype=torch.float)
        N     = x_emb.size(0)

        # === Feature extraction ===
        if self.method == "roi":
            roi_features = extract_line_feature_ROIAlign(
                img=_load_image(filepath=file_path_img, color_conversion=cv2.COLOR_BGR2RGB),
                lines=line_coords,
                output_size=self.roi_output_size,
                plot_results=True
            )
        else:  # self.method == "sample"
            # 1) load & prep image tensor
            img_np = _load_image(filepath=file_path_img, color_conversion=cv2.COLOR_BGR2RGB)
            img_t  = (
                torch.tensor(img_np, dtype=torch.float32, device=self.device)
                     .div(255.0)
                     .permute(2, 0, 1)  # C,H,W
            )

            # 2) prep lines tensor
            lines_t = torch.tensor(line_coords, dtype=torch.float32, device=self.device)

            # 3) sample
            #    returns (N, C, num_samples, width)
            roi_features = self.sampler.sample_lines_grid(
                img=img_t,
                lines=lines_t,
                align_corners=True
            )

        # sanity‐check
        if roi_features is None or roi_features.shape[0] != N:
            logging.warning(
                f"ROI feature issue for {self.filter_json_files[idx]}; "
                f"got {None if roi_features is None else roi_features.shape}, expected ({N}, …)."
            )
            raise ValueError('ROI feature extraction failed.')

        # === build graph ===
        
        coords = torch.tensor(line_coords, dtype=torch.float)

        # Use line coordinates to determine k-NN (e.g., 7 nearest neighbors)
        coords_center = np.mean(line_coords, axis=1)  # shape: (N, 2)
        nbrs = NearestNeighbors(n_neighbors=7, algorithm='auto').fit(coords_center)
        distances, indices = nbrs.kneighbors(coords_center)

        edge_list, edge_labels = [], []
        full_edge_index, full_edge_labels = [], []

        for i in range(N):
            for j in range(N):  
                full_edge_index.append([i, j])
                
            for j in indices[i]:  
                edge_list.append([i, j])
                
                
        edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()

        full_edge_index = torch.tensor(full_edge_index, dtype=torch.long).t().contiguous()
        
        # edge_list, edge_labels = [], []
        # for i in range(N):
        #     for j in range(N):
        #         edge_list.append([i, j])
        #         edge_labels.append(coplanarity_matrix[i][j])
        # edge_index  = torch.tensor(edge_list,  dtype=torch.long).t().contiguous()
        # edge_labels = torch.tensor(edge_labels, dtype=torch.float).unsqueeze(1)

        return Data(
            x=x_emb,
            edge_index=edge_index,
            geo=line_geometry(coords),

            full_edge_index=full_edge_index,
            roi_features=roi_features
        )


In [ ]:
# Paramaeters
image_pth          = 'data/ai_002_003/ai_002_003/images/scene_cam_00_final_preview/frame.0085.color.jpg'
# ckpt_pth         = "lightning_tools/lightning_logs/lightning_project_both_coplanar/euvl4yo4/checkpoints/best-model-epoch=19-val_loss_epoch=0.8419.ckpt"
ckpt_pth = "lightning_tools/lightning_logs/lightning_project/po2hi77r/checkpoints/best-model-epoch=77-val_combined_auc_epoch=0.9529.ckpt"

device            = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
color_img = cv2.imread(image_pth, cv2.IMREAD_UNCHANGED) # Load as is (handles color, grayscale, alpha)
gray_img = cv2.cvtColor(color_img, cv2.COLOR_RGB2GRAY)

plot_images([color_img], ["Image"])

In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
conf = {'detect_lines': True, 'line_detection_params': {'merge': False, 'filtering': True, 'grad_thresh': 3}}
ckpt = torch.load('../weights/deeplsd_md.tar', map_location=device, weights_only=False)
net = DeepLSD(conf)
net.load_state_dict(ckpt['model'])
net = net.to(device).eval()
    

In [ ]:
df_intermediate_features = None
angle_intermediate_features = None 
df_hook_handle = net.df_head[5].register_forward_hook(ft.hook_df)
angle_hook_handle = net.angle_head[5].register_forward_hook(ft.hook_angle)
input_tensor = torch.tensor(gray_img, dtype=torch.float32, device=device)[None, None] / 255.
with torch.no_grad():
    out = net({'image': input_tensor})
    pred_lines = out['lines'][0]
    if isinstance(pred_lines, torch.Tensor):
        pred_lines = pred_lines.cpu().numpy()
        
# get embeddings for intermediate layers.
combined_features = torch.cat([ft.df_intermediate_features, ft.angle_intermediate_features], dim=1)
downsample_ratio = color_img.shape[1] / combined_features.shape[3]
df_hook_handle.remove()
angle_hook_handle.remove()

In [ ]:
embeddings = []
coordinates = []
for i, l in enumerate(pred_lines):
    line = l.reshape(2, 2) if l.shape == (4,) else l
    coordinates.append(l.tolist())
    line_embedding = ft.sample_line_features(combined_features, line, num_samples=10, downsample_ratio=downsample_ratio).tolist()
    embeddings.append(line_embedding)

In [ ]:
dataset = GraphDatasetInference([embeddings], [image_pth], [coordinates], roi_output_size=[96, 16])

In [ ]:
sample = dataset[0]
rf = sample.roi_features               # shape: (N, C, H, W)
flat_rf = rf.view(rf.size(0), -1)      # → (N,  C*H*W )
print("dataset roi_features shape:", rf.shape)
print(" flattened ROI dim:", flat_rf.size(1))



In [ ]:
# node_preds, edge_preds = run_inference(model, dataset, model_path=model_pth, device=device)
node_preds, edge_preds = run_inference_lightning(ckpt_path=ckpt_pth, data_loader=dataset, device=device, threshold_structural=0.5, threshold_coplanarity=0.9)


In [ ]:
fig, ax = plt.subplots()

plot_lines_bool(ax, color_img, pred_lines, node_preds[0].flatten().tolist())

In [ ]:

# Convert to NumPy
edge_preds_array = edge_preds[0].numpy()
print(edge_preds_array.shape)

edge_preds_array = edge_preds_array.reshape((len(pred_lines), -1))
print(edge_preds_array.shape)

In [ ]:
import matplotlib.pyplot as plt
import math

# Parameters
batch_size = 4  # Number of subplots per figure
num_items = len(edge_preds_array)
num_batches = math.ceil(num_items / batch_size)

for batch in range(num_batches):
    start_idx = batch * batch_size
    end_idx = min(start_idx + batch_size, num_items)
    current_batch_size = end_idx - start_idx

    # Create subplots for this batch
    fig, axes = plt.subplots(nrows=1, ncols=current_batch_size, figsize=(4 * current_batch_size, 4))
    
    # Ensure axes is iterable
    if current_batch_size == 1:
        axes = [axes]
    
    for i, ax in enumerate(axes):
        idx = start_idx + i
        plot_coplanar_lines(ax, pred_lines, edge_preds_array[idx], color_img)
        ax.set_title(f'Coplanarity of line {idx + 1}')
    
    plt.tight_layout()
    plt.show()

